# Diabetes Prediction Project
### Model Workflow: Logistic Regression Implementation

This notebook tracks the predictive workflow outlined in the project report, utilizing a **Balanced Logistic Regression** pipeline to identify diabetes outcomes based on patient biometrics. 

**Workflow Steps Implemented:**
1. Data ingestion & preliminary inspection.
2. Data cleaning (replacing invalid zeros in biological indicators with standard missing values).
3. Operational train-test partitioning.
4. Data imputation using median parameters derived from training cohorts.
5. Data standardization via `StandardScaler` transformations.
6. Model execution using a class-weighted Logistic Regression configuration.

### Step 1: Ingest Libraries & Data Profile

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

# Ingesting the data
df = pd.read_csv('diabetes.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

### Step 2: Data Cleaning & Structural Handling
Biometric measurements such as `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI` cannot physically be zero. We replace these programmatic placeholders with `NaN` structural definitions to prevent skewed metrics during statistical modeling.

In [ ]:
zero_invalid_features = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[zero_invalid_features] = df[zero_invalid_features].replace(0, np.nan)

print("Missing values per column post-structural correction:")
print(df.isnull().sum())

### Step 3: Train-Test Split
To ensure reliable statistical verification, we enforce a clean feature split and partition the parameters *prior* to data imputation, preventing lookahead bias.

In [ ]:
X = df.drop(columns=['Outcome'])
y = df['Outcome']

# Enforcing train test separation 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f"Training set dimensions: {X_train.shape}")
print(f"Validation testing dimensions: {X_test.shape}")

### Step 4: Robust Data Imputation
We calculate median parameter values strictly from the training cohort to isolate calculations and perform missing value resolution safely.

In [ ]:
imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Maintain column names for downstream analysis mapping
X_train_clean = pd.DataFrame(X_train_imputed, columns=X.columns)
X_test_clean = pd.DataFrame(X_test_imputed, columns=X.columns)

### Step 5: Feature Scale Normalization
Standardizing parameter scales prevents dimensional variances from impacting the weight optimization of regularized algorithms.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_clean)
X_test_scaled = scaler.transform(X_test_clean)

### Step 6: Model Strategy Optimization & Training
We instantiate a regularized Logistic Regression classifier using `class_weight='balanced'` to offset class imbalances and intentionally optimize structural sensitivity (Recall).

In [ ]:
model = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

# Evaluate predictions
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

### Step 7: Comprehensive Model Evaluation
Extracting standard predictive metrics to verify accuracy, precision, absolute recall, and the corresponding AUC breakdown.

In [ ]:
print("--- Model Classification Matrix Report ---")
print(classification_report(y_test, y_pred))

print("--- Confusion Matrix Mapping ---")
cm = confusion_matrix(y_test, y_pred)
print(cm)

auc_metric = roc_auc_score(y_test, y_prob)
print(f"\nReceiver Operating Characteristic Curve Area (ROC-AUC): {auc_metric:.3f}")

### Step 8: Metric Visualizations (Confusion Matrix & ROC Curve)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Plotting confusion matrix heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['No Diabetes', 'Diabetes'], yticklabels=['No Diabetes', 'Diabetes'], ax=ax[0])
ax[0].set_title('Confusion Matrix Breakdown')
ax[0].set_xlabel('Predicted Label')
ax[0].set_ylabel('True Label')

# Plotting ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
ax[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {auc_metric:.3f})')
ax[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
ax[1].set_xlim([0.0, 1.0])
ax[1].set_ylim([0.0, 1.05])
ax[1].set_xlabel('False Positive Rate')
ax[1].set_ylabel('True Positive Rate')
ax[1].set_title('Receiver Operating Characteristic Curve')
ax[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

### Step 9: Feature Coefficient Mapping

In [ ]:
coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0]
}).sort_values(by='Coefficient', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x='Coefficient', y='Feature', data=coefficients, palette='viridis')
plt.title('Feature Performance Coefficients (Impact Weight)')
plt.xlabel('Coefficient Value')
plt.ylabel('Biometric Variable')
plt.show()

print(coefficients)